# Human-label evaluation: how accurate is each pose model on frames a person labelled?

Inputs: the COCO-keypoints export of the **human-labelled** `real_val` frames from CVAT (task `real_val_frames`).
These are the first numbers in this project that are **accuracy against human labels** (not agreement with an AI).
The labelled frames were never used for training. Models: COCO-pretrained, distilled, E1b, E2 (all experimental,
none integrated). The pose risk weight stays 0 whatever the result.

Report only the frames the annotator actually labelled; the count is printed in cell 3.

## 0. Inputs

In [ ]:
# 0. Drive and inputs
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive')
OUTD = DRIVE / 'pose_baseline'
HUM = OUTD / 'human'
HUM.mkdir(exist_ok=True)
HUMAN_GT = HUM / 'pose_gt_human_real_val.json'      # CVAT export "COCO Keypoints 1.0", saved here
SET_ZIP = OUTD / 'pose_annotation_set_v2.zip'
RUNS = OUTD / 'runs_pose'
MODELS = (('pretrained', OUTD / 'models' / 'yolo11s-pose.pt'),
          ('distilled', RUNS / 'yolo11s_pose_distill_960' / 'weights' / 'best.pt'),
          ('e1b', RUNS / 'yolo11s_pose_e1b_centre_960' / 'weights' / 'best.pt'),
          ('e2', RUNS / 'yolo11s_pose_e2_softconf_960' / 'weights' / 'best.pt'))
for f in [HUMAN_GT, SET_ZIP] + [w for _, w in MODELS]:
    print('OK     ' if f.is_file() else 'MISSING', f)

## 1. Install

In [ ]:
!pip -q install ultralytics==8.4.220
!mkdir -p /content/pose_tools

## 2. Tools

In [ ]:
%%writefile /content/pose_tools/pose_common.py
"""Shared helpers for the pose proposal / zero-shot tools (no project imports).

Top-down model contract (from the ViTPose-L SwimXYZ model card): input (N, 3, 256, 192) float32,
ImageNet-normalised RGB crop; output (N, 17, 64, 48) heatmaps in COCO-17 order.
Pure-numpy parts (box->crop parameters, heatmap decoding, PCK) are testable without weights.
"""
from __future__ import annotations

import math
from typing import Any

import numpy as np

COCO17 = ["nose", "left_eye", "right_eye", "left_ear", "right_ear", "left_shoulder", "right_shoulder",
          "left_elbow", "right_elbow", "left_wrist", "right_wrist", "left_hip", "right_hip",
          "left_knee", "right_knee", "left_ankle", "right_ankle"]
UPPER_BODY = [0, 5, 6, 7, 8, 9, 10]  # nose, shoulders, elbows, wrists
COCO_SKELETON = [[16, 14], [14, 12], [17, 15], [15, 13], [12, 13], [6, 12], [7, 13], [6, 7], [6, 8],
                 [7, 9], [8, 10], [9, 11], [2, 3], [1, 2], [1, 3], [2, 4], [3, 5], [4, 6], [5, 7]]
IN_W, IN_H = 192, 256
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


def crop_params(box_xyxy: tuple[float, float, float, float], scale: float = 1.25) -> tuple[float, float, float]:
    """Return (x0, y0, s) of the crop window: window covers IN_W/s x IN_H/s pixels, box centred, padded."""
    x1, y1, x2, y2 = box_xyxy
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    bw, bh = max(x2 - x1, 1.0) * scale, max(y2 - y1, 1.0) * scale
    if bw / bh > IN_W / IN_H:
        bh = bw * IN_H / IN_W
    else:
        bw = bh * IN_W / IN_H
    s = IN_W / bw
    return cx - bw / 2, cy - bh / 2, s


def preprocess(frame_bgr: np.ndarray, box_xyxy: tuple[float, float, float, float]
               ) -> tuple[np.ndarray, tuple[float, float, float]]:
    import cv2  # lazy: Colab has it

    x0, y0, s = crop_params(box_xyxy)
    m = np.array([[s, 0, -x0 * s], [0, s, -y0 * s]], dtype=np.float32)
    crop = cv2.warpAffine(frame_bgr, m, (IN_W, IN_H), flags=cv2.INTER_LINEAR,
                          borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
    rgb = crop[:, :, ::-1].astype(np.float32) / 255.0
    rgb = (rgb - MEAN) / STD
    return rgb.transpose(2, 0, 1)[None], (x0, y0, s)


def decode_heatmaps(hm: np.ndarray, params: tuple[float, float, float]) -> np.ndarray:
    """hm: (17, H, W) -> (17, 3) [x, y, score] in image pixels (argmax, +quarter-pixel shift)."""
    k, h, w = hm.shape
    x0, y0, s = params
    out = np.zeros((k, 3), dtype=np.float32)
    for i in range(k):
        flat = int(np.argmax(hm[i]))
        iy, ix = divmod(flat, w)
        score = float(hm[i, iy, ix])
        fx, fy = float(ix), float(iy)
        if 0 < ix < w - 1:
            fx += 0.25 * np.sign(hm[i, iy, ix + 1] - hm[i, iy, ix - 1])
        if 0 < iy < h - 1:
            fy += 0.25 * np.sign(hm[i, iy + 1, ix] - hm[i, iy - 1, ix])
        cx, cy = (fx + 0.5) * IN_W / w, (fy + 0.5) * IN_H / h  # crop pixels
        out[i] = (cx / s + x0, cy / s + y0, score)
    return out


class OnnxVitPose:
    """Top-down ONNX pose model (ViTPose-L SwimXYZ export). Input name is read from the graph."""

    def __init__(self, path: str, providers: list[str] | None = None) -> None:
        import onnxruntime as ort

        self.sess = ort.InferenceSession(path, providers=providers or ["CUDAExecutionProvider",
                                                                         "CPUExecutionProvider"])
        self.input_name = self.sess.get_inputs()[0].name

    def predict(self, frame_bgr: np.ndarray, boxes: list[tuple[float, float, float, float]]) -> list[np.ndarray]:
        res = []
        for b in boxes:
            x, params = preprocess(frame_bgr, b)
            hm = self.sess.run(None, {self.input_name: x})[0][0]
            res.append(decode_heatmaps(hm, params))
        return res


def iou(a: tuple[float, ...], b: tuple[float, ...]) -> float:
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def pck(pred: np.ndarray, gt: np.ndarray, box_xyxy: tuple[float, ...], thr: float,
        idx: list[int] | None = None) -> tuple[int, int]:
    """(correct, total) over GT keypoints with visibility > 0. Normaliser: longer side of the GT box."""
    norm = max(box_xyxy[2] - box_xyxy[0], box_xyxy[3] - box_xyxy[1], 1.0)
    ok = tot = 0
    for i in idx if idx is not None else range(len(gt)):
        if gt[i][2] <= 0:
            continue
        tot += 1
        if math.hypot(pred[i][0] - gt[i][0], pred[i][1] - gt[i][1]) <= thr * norm:
            ok += 1
    return ok, tot


def confident_upper(pred: np.ndarray, min_conf: float = 0.5) -> int:
    return int(sum(1 for i in UPPER_BODY if pred[i][2] >= min_conf))


def as_info(**kw: Any) -> dict[str, Any]:
    return kw

In [ ]:
%%writefile /content/pose_tools/zero_shot_eval.py
"""Zero-shot comparison of pose models on HAND-CORRECTED frames (CVAT export, COCO keypoints 1.0).

Metrics (all on GT keypoints with visibility > 0; normaliser = longer side of the GT box):
  PCK@0.05 / 0.1 / 0.2 on upper body (nose, shoulders, elbows, wrists) and on all 17,
  share of frames with >= 3 confident (>=0.5) upper-body keypoints, latency per person.
Models (any subset):
  --vitpose-onnx   ViTPose-L SwimXYZ ONNX, top-down on the GT box
  --yolo-pose      Ultralytics pose weights, full frame; the prediction with best IoU to the GT box is used
Only run on the `real_val` frames; select with --split-manifest + --split real_val. Report per camera type via --group.
"""
from __future__ import annotations

import argparse
import csv
import json
import time
from pathlib import Path

import cv2
import numpy as np

from pose_common import UPPER_BODY, OnnxVitPose, confident_upper, iou, pck

THRS = (0.05, 0.1, 0.2)
FLIP_IDX = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]  # COCO-17 left <-> right


def load_gt(coco_path: Path) -> dict[str, tuple[np.ndarray, tuple[float, float, float, float]]]:
    d = json.loads(coco_path.read_text(encoding="utf-8"))
    names = {i["id"]: i["file_name"] for i in d["images"]}
    out = {}
    for a in d["annotations"]:
        k = np.array(a["keypoints"], dtype=np.float32).reshape(-1, 3)
        if k.shape[0] != 17:
            raise SystemExit(f"annotation {a['id']} has {k.shape[0]} keypoints, expected 17 (COCO order)")
        x, y, w, h = a["bbox"]
        out[Path(names[a["image_id"]]).name] = (k, (x, y, x + w, y + h))
    return out


def run_vitpose(model: OnnxVitPose, img: np.ndarray, box: tuple[float, ...]) -> np.ndarray:
    return model.predict(img, [box])[0]


def run_yolo(model, img: np.ndarray, box: tuple[float, ...]) -> np.ndarray | None:  # noqa: ANN001
    res = model.predict(img, verbose=False)[0]
    if res.keypoints is None or len(res.boxes) == 0:
        return None
    best, best_iou = None, 0.0
    for i, b in enumerate(res.boxes.xyxy.cpu().numpy()):
        v = iou(tuple(b), box)
        if v > best_iou:
            best, best_iou = i, v
    if best is None or best_iou < 0.3:
        return None
    xy = res.keypoints.xy[best].cpu().numpy()
    cf = res.keypoints.conf[best].cpu().numpy() if res.keypoints.conf is not None else np.ones(17)
    return np.concatenate([xy, cf[:, None]], axis=1)


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__)
    ap.add_argument("--gt", type=Path, required=True, help="CVAT export (COCO keypoints 1.0) json")
    ap.add_argument("--images", type=Path, required=True, help="folder with the images of the set")
    ap.add_argument("--manifest", type=Path, help="manifest.csv, to restrict to --split")
    ap.add_argument("--split", default="real_val")
    ap.add_argument("--group", default="", help="substring of image name defining a camera group, e.g. 'ddt'")
    ap.add_argument("--vitpose-onnx", type=str)
    ap.add_argument("--yolo-pose", type=str)
    ap.add_argument("--out", type=Path, default=Path("zero_shot_results.csv"))
    ap.add_argument("--boxes-from-manifest", action="store_true",
                    help="use the manifest person boxes (x1..y2) instead of the GT bbox. Use for CVAT exports, whose "
                         "bbox is fitted tightly around the annotated points")
    a = ap.parse_args()

    if not a.gt.is_file():
        raise SystemExit(f"GT file not found: {a.gt}. Export the corrected real_val frames from CVAT first "
                         "(see docs/POSE_ANNOTATION_GUIDE.md); this step cannot run before that.")
    gt = load_gt(a.gt)
    if a.boxes_from_manifest:
        if not a.manifest:
            raise SystemExit("--boxes-from-manifest needs --manifest")
        mboxes = {Path(r["image"]).name: tuple(float(r[k]) for k in ("x1", "y1", "x2", "y2"))
                  for r in csv.DictReader(open(a.manifest, encoding="utf-8"))}
        gt = {n: (k, mboxes[n]) for n, (k, _) in gt.items() if n in mboxes}
    names = set(gt)
    if a.manifest:
        names &= {Path(r["image"]).name for r in csv.DictReader(open(a.manifest, encoding="utf-8"))
                  if r["split"] == a.split}
    if a.group:
        names = {n for n in names if a.group in n}
    names = sorted(names)
    if not names:
        raise SystemExit("no frames selected: check --split / --group / file names")
    print(f"{len(names)} frames, split={a.split}, group={a.group or 'all'}")

    runners = {}
    if a.vitpose_onnx:
        m = OnnxVitPose(a.vitpose_onnx)
        runners["vitpose_swimxyz"] = lambda im, b: run_vitpose(m, im, b)
    if a.yolo_pose:
        from ultralytics import YOLO

        y = YOLO(a.yolo_pose)
        runners[Path(a.yolo_pose).stem] = lambda im, b: run_yolo(y, im, b)
    if not runners:
        raise SystemExit("give --vitpose-onnx and/or --yolo-pose")

    rows = []
    for mname, fn in runners.items():
        acc = {(s, t): [0, 0] for s in ("upper", "all") for t in THRS}
        acc_sw = {(s, t): [0, 0] for s in ("upper", "all") for t in THRS}
        n_conf3 = n_missing = n_swapped = 0
        times = []
        for n in names:
            img = cv2.imread(str(a.images / n))
            k, box = gt[n]
            t0 = time.perf_counter()
            pred = fn(img, box)
            times.append((time.perf_counter() - t0) * 1000)
            if pred is None:
                n_missing += 1
                pred = np.zeros((17, 3), dtype=np.float32)
                pred[:, :2] = -1e6
            n_conf3 += confident_upper(pred) >= 3
            # swap-tolerant: one left/right decision per frame (whole skeleton mirrored in labels, positions kept),
            # chosen by the larger PCK@0.2 over all keypoints; ties keep the original labelling
            swapped = pred[FLIP_IDX]
            use_swap = pck(swapped, k, box, 0.2)[0] > pck(pred, k, box, 0.2)[0]
            n_swapped += use_swap
            best = swapped if use_swap else pred
            for t in THRS:
                for s, idx in (("upper", UPPER_BODY), ("all", None)):
                    ok, tot = pck(pred, k, box, t, idx)
                    acc[(s, t)][0] += ok
                    acc[(s, t)][1] += tot
                    ok2, _ = pck(best, k, box, t, idx)
                    acc_sw[(s, t)][0] += ok2
                    acc_sw[(s, t)][1] += tot
        row = {"model": mname, "frames": len(names), "no_prediction": n_missing,
               "frames_ge3_confident_upper": round(n_conf3 / len(names), 3),
               "latency_ms_median_per_person": round(float(np.median(times)), 1),
               "frames_better_with_lr_swap": n_swapped}
        for (s, t), (ok, tot) in acc.items():
            row[f"PCK@{t}_{s}"] = round(ok / tot, 3) if tot else None
        for (s, t), (ok, tot) in acc_sw.items():
            row[f"PCK@{t}_{s}_swaptol"] = round(ok / tot, 3) if tot else None
        rows.append(row)
        print(row)
    with open(a.out, "w", newline="", encoding="utf-8") as f:
        wr = csv.DictWriter(f, fieldnames=list(rows[0]))
        wr.writeheader()
        wr.writerows(rows)
    print("wrote", a.out)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
%%writefile /content/pose_tools/conf_calibration.py
"""Does a pose model's keypoint confidence carry information? (E2 check; agreement with reference labels, not accuracy)

For every labelled reference keypoint, a point is a HIT if the model's point (after one left/right decision per frame,
as in the swap-tolerant PCK) is within --thr of the box's longer side. The script reports the mean confidence of hits and
misses and the AUROC of confidence for separating them. AUROC 0.5 = confidence says nothing (E1b's always-high
confidence); higher is better. Frames without a matched detection count as misses with confidence 0.
"""
from __future__ import annotations

import argparse
import csv
from pathlib import Path

import cv2
import numpy as np

from pose_common import iou
from zero_shot_eval import load_gt

FLIP_IDX = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]


def auroc(scores: np.ndarray, labels: np.ndarray) -> float:
    """Rank-based AUROC (ties get average rank); nan if only one class is present."""
    pos, neg = labels.sum(), (~labels).sum()
    if pos == 0 or neg == 0:
        return float("nan")
    order = scores.argsort(kind="mergesort")
    ranks = np.empty(len(scores))
    s = scores[order]
    i = 0
    while i < len(s):
        j = i
        while j + 1 < len(s) and s[j + 1] == s[i]:
            j += 1
        ranks[order[i:j + 1]] = (i + j) / 2 + 1
        i = j + 1
    return float((ranks[labels].sum() - pos * (pos + 1) / 2) / (pos * neg))


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--gt", type=Path, required=True, help="COCO keypoint reference JSON")
    ap.add_argument("--images", type=Path, required=True)
    ap.add_argument("--manifest", type=Path, help="restrict to --split (and optionally use its boxes)")
    ap.add_argument("--split")
    ap.add_argument("--boxes-from-manifest", action="store_true")
    ap.add_argument("--weights", nargs="+", required=True, help="name=path.pt entries")
    ap.add_argument("--thr", type=float, default=0.1)
    ap.add_argument("--out", type=Path)
    a = ap.parse_args()

    from ultralytics import YOLO

    gt = load_gt(a.gt)
    names = set(gt)
    rows = list(csv.DictReader(open(a.manifest, encoding="utf-8"))) if a.manifest else []
    if a.manifest:
        names &= {Path(r["image"]).name for r in rows if not a.split or r["split"] == a.split}
    if a.boxes_from_manifest:
        mb = {Path(r["image"]).name: tuple(float(r[k]) for k in ("x1", "y1", "x2", "y2")) for r in rows}
        gt = {n: (k, mb[n]) for n, (k, _) in gt.items() if n in mb}
    names = sorted(n for n in names if n in gt)

    results = []
    for spec in a.weights:
        tag, w = spec.split("=", 1)
        m = YOLO(w)
        confs, hits = [], []
        for n in names:
            ref, box = gt[n]
            lab = ref[:, 2] > 0
            size = max(box[2] - box[0], box[3] - box[1], 1.0)
            r = m.predict(cv2.imread(str(a.images / n)), verbose=False)[0]
            pred = None
            if r.keypoints is not None and len(r.boxes):
                ious = [iou(tuple(b), box) for b in r.boxes.xyxy.cpu().numpy()]
                i = int(np.argmax(ious))
                if ious[i] >= 0.3:
                    pred = np.concatenate([r.keypoints.xy[i].cpu().numpy(), r.keypoints.conf[i].cpu().numpy()[:, None]], 1)
            if pred is None:
                confs += [0.0] * int(lab.sum()); hits += [False] * int(lab.sum())
                continue
            def hit_of(p):
                return np.hypot(*(p[:, :2] - ref[:, :2]).T) / size <= a.thr
            h0, h1 = hit_of(pred), hit_of(pred[FLIP_IDX])
            if h1[lab].sum() > h0[lab].sum():
                pred, h0 = pred[FLIP_IDX], h1
            confs += pred[lab, 2].tolist(); hits += h0[lab].tolist()
        c, h = np.array(confs), np.array(hits, dtype=bool)
        res = {"model": tag, "frames": len(names), "points": len(c), "hit_rate": round(float(h.mean()), 3),
               "mean_conf_hits": round(float(c[h].mean()), 3) if h.any() else float("nan"),
               "mean_conf_misses": round(float(c[~h].mean()), 3) if (~h).any() else float("nan"),
               "conf_auroc": round(auroc(c, h), 3)}
        results.append(res)
        print(res)
    if a.out:
        with open(a.out, "w", newline="", encoding="utf-8") as f:
            wr = csv.DictWriter(f, fieldnames=list(results[0])); wr.writeheader(); wr.writerows(results)
        print("wrote", a.out)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

## 3. Data and what was labelled

In [ ]:
# 3. Annotation set + a check of the human export (provenance, number of labelled frames and points)
import json, zipfile, csv, collections
if not Path('/content/pose_annotation_set_v2').is_dir():
    with zipfile.ZipFile(SET_ZIP) as z:
        z.extractall('/content')
ASET = Path('/content/pose_annotation_set_v2')
d = json.load(open(HUMAN_GT, encoding='utf-8'))
man = {Path(r['image']).name: r for r in csv.DictReader(open(ASET / 'manifest.csv', encoding='utf-8'))}
names = {i['id']: Path(i['file_name']).name for i in d['images']}
lab = collections.Counter()
for a in d['annotations']:
    k = a['keypoints']; n = sum(1 for v in k[2::3] if v > 0)
    if n: lab[names[a['image_id']]] = n
splits = collections.Counter(man[n]['split'] for n in lab if n in man)
print('frames with >= 1 labelled point:', len(lab), '| by split:', dict(splits))
print('labelled points total:', sum(lab.values()), '| mean per frame: %.1f' % (sum(lab.values()) / max(len(lab), 1)))
print('NOTE: only real_val frames are used below; real_train frames in the export (if any) are ignored.')

## 4. Accuracy against the human labels

In [ ]:
# 4. PCK against HUMAN labels (frames the annotator labelled, split real_val). One row per model.
import pandas as pd
for tag, w in MODELS:
    !cd /content/pose_tools && python zero_shot_eval.py --gt "{HUMAN_GT}" --images "{ASET}/images" --manifest "{ASET}/manifest.csv" --split real_val --boxes-from-manifest --yolo-pose "{w}" --out "{HUM}/human_{tag}.csv" | grep wrote
tab = pd.concat([pd.read_csv(HUM / f'human_{t}.csv').assign(model=t) for t, _ in MODELS])
tab.to_csv(HUM / 'human_comparison.csv', index=False)
print('ACCURACY AGAINST HUMAN LABELS (annotator-labelled real_val frames)')
print(tab.set_index('model').T.to_string())

## 5. Does the confidence carry information? (against the human labels)

In [ ]:
# 5. AUROC of confidence for separating hits from misses (0.5 = confidence says nothing), hit = within 0.1 of box size
W = ' '.join(f'{t}={w}' for t, w in MODELS)
!cd /content/pose_tools && python conf_calibration.py --gt "{HUMAN_GT}" --images "{ASET}/images" --manifest "{ASET}/manifest.csv" --split real_val --boxes-from-manifest --weights {W} --out "{HUM}/human_calibration.csv" | grep -v wrote

## How to read it
- Compare the models on `PCK@0.1_upper` and `PCK@0.1_upper_swaptol` first (head, shoulders, elbows, wrists). The
  swap-tolerant rows ignore left/right mix-ups; the plain rows count them.
- A useful supporting signal needs clearly better-than-chance positions **and** a confidence AUROC well above 0.5.
- With only the annotator's frames, differences of a few points are noise. Save the notebook after cell 5.